# Object-summary batch processing

Summarize every Markdown file under these two corpora, in order:

- `PROJ_ROOT/assets/md.json.semantics.coding.4.0`
- `PROJ_ROOT/assets/md.json.semantics.coding.5.0`

Uses `ribosome.core.dom.summary.object.summarize_markdown_object_tree_async`.
Figures, HTML/Markdown pipe tables, and lists remain whole objects with their
captions and original subtrees. Sections, prose, and parent/document summaries
remain in the recursive DOM tree. Successful documents atomically replace their
adjacent `<markdown_stem>_semantics.json`.

This notebook is saved without outputs. Run cells in order to start processing;
creating or opening it does not run the corpus. Embedding and ingestion are excluded.

## Configuration

Use `OLLAMA_HOST`, `OLLAMA_SUMMARY_MODEL`, and optional `OLLAMA_API_KEY` from
`PROJ_ROOT/.env`. One image-capable Ollama model handles both text and figures.
The notebook does not modify `.env`.

Set limits in the environment or `.env` before running this cell. Defaults are
**4 document workers, 8 shared model requests, and 32 traversal tasks per document**.
Limits are applied before the object summarizer is imported. If it is already
loaded with different limits, restart the kernel and run cells in order.
Markdown is parsed directly; existing AST artifacts are left untouched.

In [ ]:
import asyncio
import json
import os
import sys
from pathlib import Path
from time import perf_counter

from dotenv import dotenv_values, load_dotenv


def resolve_project_root():
    configured = os.getenv("PROJ_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "pyproject.toml").is_file():
            raise FileNotFoundError(f"Invalid PROJ_ROOT (missing pyproject.toml): {root}")
        return root
    cwd = Path.cwd().resolve()
    for root in (cwd, *cwd.parents):
        if (root / "pyproject.toml").is_file():
            return root
    raise FileNotFoundError("Cannot locate PROJ_ROOT containing pyproject.toml")


PROJ_ROOT = resolve_project_root()
ENV_FILE = PROJ_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Missing configuration: {ENV_FILE}")
load_dotenv(ENV_FILE, override=False)
env_config = dotenv_values(ENV_FILE)
OLLAMA_HOST = (env_config.get("OLLAMA_HOST") or "").strip()
SUMMARY_MODEL = (env_config.get("OLLAMA_SUMMARY_MODEL") or "").strip()
if not OLLAMA_HOST or not SUMMARY_MODEL:
    raise ValueError("Set OLLAMA_HOST and OLLAMA_SUMMARY_MODEL in PROJ_ROOT/.env")
os.environ["OLLAMA_HOST"] = OLLAMA_HOST
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

from utils.summary_asyncio import configure_summary_limits, SummaryAsyncClient
SUMMARY_LIMITS = configure_summary_limits(summary_module="object")
DOCUMENT_CONCURRENCY = SUMMARY_LIMITS["RIBOSOME_DOCUMENT_CONCURRENCY"]

import pandas as pd
from IPython.display import display
from utils.summary_object_asyncio import discover_markdown, check_model, summarize_object_documents

INPUT_DIRS = [
    PROJ_ROOT / "assets" / "md.json.semantics.coding.4.0",
    PROJ_ROOT / "assets" / "md.json.semantics.coding.5.0",
]
CHECKPOINT_NAME = ".object-summary-checkpoint.json"
RESUME = True  # Set False after changing images/summarizer code or to rebuild every document.
LANG = "zh"
LEAF_MIN_LEN = 100
MIN_LEN = None
api_key = env_config.get("OLLAMA_API_KEY")
headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
ollama_client = SummaryAsyncClient(host=OLLAMA_HOST, headers=headers)
print("Inputs:")
for input_dir in INPUT_DIRS:
    print(f"  {input_dir}")
print(f"Ollama: {OLLAMA_HOST}\nText and image model: {SUMMARY_MODEL}")
print(f"Effective concurrency: {SUMMARY_LIMITS}")

## Discovery and provider preflight

Recursively discover sorted `.md` files in both corpora, including nested paths
and uppercase `.MD` extensions. Missing or empty directories stop preflight.
All documents are included even if the same manual exists in both versions.
The configured model must already be available on the server. This step does
not generate summaries or write artifacts.

In [ ]:
corpora = [(input_dir, discover_markdown(input_dir)) for input_dir in INPUT_DIRS]
for input_dir, md_files in corpora:
    print(f"{input_dir.name}: {len(md_files)} Markdown files")
print(f"Total Markdown files: {sum(len(files) for _, files in corpora)}")
await check_model(ollama_client, SUMMARY_MODEL, OLLAMA_HOST)
print(f"Provider ready: {SUMMARY_MODEL}")

## Rerun object summarization

Process 4.0 first, followed by 5.0, with bounded concurrent documents within each
corpus. Each successful document replaces its adjacent `_semantics.json` only
after the complete tree and its document summary are ready. Failed or cancelled
documents keep their previous output. Other documents continue after failures.

Progress is atomically checkpointed in each corpus's `.object-summary-checkpoint.json`,
separately from natural-text checkpoints. With `RESUME=True`, unchanged successful
documents are skipped. Changes to Markdown, cached-artifact signatures, output,
model, host, language, or summary thresholds cause reprocessing. Failed or
interrupted documents restart from the beginning. Changes to images or summarizer
code require `RESUME=False`. Keep checkpoints alongside their corpora.

Two progress bars show completed documents and objects. Displays refresh every
**250ms**, with a final flush; completed objects may still be followed by section
and document-summary requests. Request limits are shared across text and images.
Interrupting cancels outstanding model work, awaits cleanup, and saves the
completed-document checkpoint. Synchronous Markdown/Pandoc parsing finishes
before cancellation can be observed.

In [ ]:
ollama_client.reset_metrics()
batch_start = perf_counter()
results = []
for input_dir, md_files in corpora:
    print(f"Processing corpus: {input_dir.name}")
    corpus_results = await summarize_object_documents(
        md_files, ollama_client, SUMMARY_MODEL, corpus=input_dir.name,
        concurrency=DOCUMENT_CONCURRENCY, progress_interval=0.25,
        checkpoint_path=input_dir / CHECKPOINT_NAME,
        resume=RESUME, checkpoint_host=OLLAMA_HOST,
        lang=LANG, leaf_min_len=LEAF_MIN_LEN, min_len=MIN_LEN,
    )
    results.extend(corpus_results)
batch_elapsed_seconds = perf_counter() - batch_start
batch_metrics = ollama_client.metrics.snapshot(wall_seconds=batch_elapsed_seconds)

## Results

Review failures before rerunning. Unchanged successful documents are marked
`resumed=True`; their timings and request metrics come from their saved run.
Batch metrics describe only requests made during this run. Output paths identify
the complete recursive JSON trees. `object_count` includes figures, tables, and lists;
`completed_objects` counts successful whole-object summaries even if a later
parent summary fails.

In [ ]:
results_df = pd.DataFrame(results)
display(results_df)
if not results_df.empty:
    print(results_df.groupby(["corpus", "status"]).size().to_string())
    print(f"Summed document processing time: {results_df['elapsed_seconds'].sum():.3f}s")
    failures = results_df.loc[results_df["status"] == "failed"]
    if not failures.empty:
        print("Failed documents:")
        display(failures[["corpus", "file", "error"]])
print(f"Batch wall time: {batch_elapsed_seconds:.3f}s")
display(pd.DataFrame([batch_metrics]))

## Performance and reruns

The corpora contain thousands of objects; figures may require an image request
for every image plus a combined summary request. Parent and document summaries
also contribute to runtime. Compare client latency, Ollama prompt/generation
timings, and peak requests in the results before adjusting concurrency.
Client request overlap is not the same as server inference parallelism.

Change concurrency settings before importing the object module and restart the
kernel when prompted. Use `RESUME=True` to continue after an interruption, and
`RESUME=False` to rebuild after image or summarizer changes. Resetting resume
replaces the object checkpoint; it does not alter the Markdown or AST cache.

The natural-text benchmark utility measures a different pipeline and is not
used by this notebook. No corpus processing is performed during notebook creation
or offline validation.